[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu231204/pytorch-mastery/blob/main/01_fundamentals/datasets_dataloaders.ipynb)



# 04 . Datasets and DataLoaders

## Concept — Dataset & DataLoader (revision notes)

**What they are**
- `Dataset` — an object that knows how to fetch ONE sample at a time, given
  an index. It answers "what is item `i`?" and "how many items are there?"
- `DataLoader` — wraps a `Dataset` and handles batching, shuffling, and
  (optionally) parallel loading with multiple worker processes.
- Together: `Dataset` defines *what* the data is, `DataLoader` defines *how*
  you iterate over it during training.

**Why they exist**
- Training needs data fed in mini-batches, shuffled every epoch, ideally
  loaded in parallel with the GPU still computing on the previous batch.
  Writing this by hand every time is repetitive and error-prone —
  `DataLoader` handles it consistently for any `Dataset`.

**The two Dataset styles**
- **Map-style** (most common): subclass `torch.utils.data.Dataset`, implement
  `__len__` and `__getitem__(self, idx)`. Supports shuffling, random access,
  and `len()`.
- **Iterable-style**: subclass `torch.utils.data.IterableDataset`, implement
  `__iter__`. Used for streaming data (e.g. reading from a huge file or a
  live feed) where random access by index doesn't make sense.

**What `DataLoader` actually does for you**
- **Batching** — groups individual samples from `Dataset.__getitem__` into
  batches of a given `batch_size`.
- **Shuffling** — `shuffle=True` reshuffles the order every epoch (should be
  `True` for training, `False` for validation/test).
- **Parallel loading** — `num_workers > 0` spins up separate worker
  processes to fetch/preprocess samples while the GPU is busy, avoiding a
  data-loading bottleneck.
- **Collation** — the `collate_fn` decides how a list of individual samples
  becomes one batched tensor (or tuple of tensors). The default works for
  same-shaped tensors; you write a custom one when samples have variable
  length (e.g. sentences of different lengths).

**Key arguments to know**
- `batch_size` — how many samples per batch.
- `shuffle` — reshuffle order every epoch (training: `True`, eval: `False`).
- `num_workers` — how many subprocesses do the loading; `0` means load in
  the main process (simplest, sometimes slowest).
- `pin_memory=True` — speeds up the CPU→GPU transfer; use when training on GPU.
- `drop_last` — whether to drop the final incomplete batch if the dataset
  size isn't evenly divisible by `batch_size`.
- `collate_fn` — custom function to combine a list of samples into a batch;
  essential for variable-length data like text sequences.

**Samplers — controlling *which* order/subset**
- `DataLoader`'s `shuffle=True` is actually shorthand for using a
  `RandomSampler` internally.
- `SubsetRandomSampler` — train/val split from one dataset without copying data.
- `WeightedRandomSampler` — oversample rare classes, a common technique for
  imbalanced datasets.
- You pass a custom `sampler=` instead of `shuffle=` when you need this
  level of control (they're mutually exclusive arguments).

**Common real-world pattern**
- Load raw data once in `Dataset.__init__` (or index it lazily for huge
  datasets), do per-sample transforms in `__getitem__` (e.g. image
  augmentation), and let `DataLoader` handle the batching machinery.


### 1. A minimal custom Dataset

**What this cell does (plain language):** we build the simplest possible
custom dataset — wrapping a list of numbers and their labels — implementing
just the two required methods, `__len__` and `__getitem__`. Then we access a
few items directly to see exactly what a `Dataset` returns per index.


In [1]:
import torch
from torch.utils.data import Dataset, DataLoader

class NumberDataset(Dataset):
    def __init__(self, numbers, labels):
        self.numbers = numbers
        self.labels = labels

    def __len__(self):
        # DataLoader calls this to know how many samples exist / when an epoch ends
        return len(self.numbers)

    def __getitem__(self, idx):
        # DataLoader calls this with an index to fetch ONE sample
        x = torch.tensor([self.numbers[idx]], dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

numbers = list(range(10))
labels = [n % 2 for n in numbers]  # even/odd as a toy classification target

dataset = NumberDataset(numbers, labels)
print("dataset length:", len(dataset))
print("dataset[0]:", dataset[0])
print("dataset[5]:", dataset[5])


dataset length: 10
dataset[0]: (tensor([0.]), tensor(0))
dataset[5]: (tensor([5.]), tensor(1))


### 2. Wrapping it in a DataLoader — batching and shuffling

**What this cell does (plain language):** we wrap the dataset from above in
a `DataLoader` with `batch_size=4` and iterate over it, printing each batch
so you can see individual samples get grouped together automatically. We
compare `shuffle=False` (same order every time) against `shuffle=True`
(different order each time you iterate).


In [2]:
loader_no_shuffle = DataLoader(dataset, batch_size=4, shuffle=False)
print("shuffle=False, epoch 1:")
for batch_x, batch_y in loader_no_shuffle:
    print("  x:", batch_x.squeeze().tolist(), "| y:", batch_y.tolist())

print("\nshuffle=False, epoch 2 (should be IDENTICAL order):")
for batch_x, batch_y in loader_no_shuffle:
    print("  x:", batch_x.squeeze().tolist(), "| y:", batch_y.tolist())

loader_shuffle = DataLoader(dataset, batch_size=4, shuffle=True)
print("\nshuffle=True, epoch 1:")
for batch_x, batch_y in loader_shuffle:
    print("  x:", batch_x.squeeze().tolist(), "| y:", batch_y.tolist())

print("\nshuffle=True, epoch 2 (order should DIFFER from epoch 1):")
for batch_x, batch_y in loader_shuffle:
    print("  x:", batch_x.squeeze().tolist(), "| y:", batch_y.tolist())


shuffle=False, epoch 1:
  x: [0.0, 1.0, 2.0, 3.0] | y: [0, 1, 0, 1]
  x: [4.0, 5.0, 6.0, 7.0] | y: [0, 1, 0, 1]
  x: [8.0, 9.0] | y: [0, 1]

shuffle=False, epoch 2 (should be IDENTICAL order):
  x: [0.0, 1.0, 2.0, 3.0] | y: [0, 1, 0, 1]
  x: [4.0, 5.0, 6.0, 7.0] | y: [0, 1, 0, 1]
  x: [8.0, 9.0] | y: [0, 1]

shuffle=True, epoch 1:
  x: [5.0, 3.0, 1.0, 7.0] | y: [1, 1, 1, 1]
  x: [9.0, 2.0, 8.0, 4.0] | y: [1, 0, 0, 0]
  x: [0.0, 6.0] | y: [0, 0]

shuffle=True, epoch 2 (order should DIFFER from epoch 1):
  x: [3.0, 8.0, 5.0, 4.0] | y: [1, 0, 1, 0]
  x: [6.0, 1.0, 7.0, 2.0] | y: [0, 1, 1, 0]
  x: [0.0, 9.0] | y: [0, 1]


### 3. drop_last — what happens when the dataset doesn't divide evenly

**What this cell does (plain language):** our dataset has 10 samples, and a
batch size of 4 doesn't divide evenly into it (10 / 4 = 2 full batches + 2
leftover). We compare `drop_last=False` (keeps the small final batch) vs
`drop_last=True` (throws it away) so you can see the size of the last batch
in each case.


In [ ]:
loader_keep = DataLoader(dataset, batch_size=4, shuffle=False, drop_last=False)
print("drop_last=False, batch sizes:", [len(b[0]) for b in loader_keep])

loader_drop = DataLoader(dataset, batch_size=4, shuffle=False, drop_last=True)
print("drop_last=True,  batch sizes:", [len(b[0]) for b in loader_drop])

# drop_last=True matters most when your model/training step assumes a FIXED
# batch size (e.g. certain BatchNorm configurations, or some distributed
# training setups) - an inconsistent final batch size can cause errors there.


drop_last=False, batch sizes: [4, 4, 2]
drop_last=True,  batch sizes: [4, 4]


### 4. Custom collate_fn — for variable-length data

**What this cell does (plain language):** real text/sequence data often has
different lengths per sample, so you can't just stack them into one tensor
like fixed-size numbers. We build a toy "sentence" dataset (lists of
different lengths) and write a custom `collate_fn` that pads each batch to
the same length before stacking.


In [4]:
class ToySentenceDataset(Dataset):
    def __init__(self, sentences):
        self.sentences = sentences  # list of lists of token ids, variable length

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        return torch.tensor(self.sentences[idx], dtype=torch.long)

sentences = [
    [1, 2, 3],
    [4, 5],
    [6, 7, 8, 9, 10],
    [11],
]
sentence_dataset = ToySentenceDataset(sentences)

def pad_collate(batch):
    # batch is a list of 1D tensors of different lengths
    lengths = [len(seq) for seq in batch]
    max_len = max(lengths)
    padded = torch.zeros(len(batch), max_len, dtype=torch.long)  # 0 = padding token
    for i, seq in enumerate(batch):
        padded[i, :len(seq)] = seq
    return padded, torch.tensor(lengths)

sentence_loader = DataLoader(sentence_dataset, batch_size=2, shuffle=False, collate_fn=pad_collate)
for padded_batch, lengths in sentence_loader:
    print("padded batch:\n", padded_batch)
    print("original lengths:", lengths)
    print()

# Without a custom collate_fn, DataLoader's default tries to torch.stack()
# the samples directly, which FAILS when they have different shapes.


padded batch:
 tensor([[1, 2, 3],
        [4, 5, 0]])
original lengths: tensor([3, 2])

padded batch:
 tensor([[ 6,  7,  8,  9, 10],
        [11,  0,  0,  0,  0]])
original lengths: tensor([5, 1])



### 5. What happens without a custom collate_fn (the error)

**What this cell does (plain language):** we deliberately try to load the
same variable-length sentence dataset WITHOUT a custom `collate_fn`, to show
you the exact error message you'll get in real projects, so you recognize it
immediately.


In [5]:
default_loader = DataLoader(sentence_dataset, batch_size=2, shuffle=False)  # no collate_fn
try:
    for batch in default_loader:
        print(batch)
except RuntimeError as e:
    print("Expected error:", str(e)[:150])


Expected error: stack expects each tensor to be equal size, but got [3] at entry 0 and [2] at entry 1


### 6. Splitting a dataset into train/val with SubsetRandomSampler

**What this cell does (plain language):** instead of creating two separate
`Dataset` objects, we use `SubsetRandomSampler` to tell one `DataLoader`
"only use these specific indices" — a common way to do a train/val split
without duplicating any data in memory.


In [6]:
from torch.utils.data import SubsetRandomSampler
import random

full_dataset = NumberDataset(list(range(20)), [n % 2 for n in range(20)])

indices = list(range(len(full_dataset)))
random.Random(42).shuffle(indices)  # fixed seed for reproducibility
split = int(0.8 * len(indices))
train_indices, val_indices = indices[:split], indices[split:]

train_sampler = SubsetRandomSampler(train_indices)
val_sampler = SubsetRandomSampler(val_indices)

# NOTE: when using a sampler, do NOT also pass shuffle= - they're mutually exclusive
train_loader = DataLoader(full_dataset, batch_size=4, sampler=train_sampler)
val_loader = DataLoader(full_dataset, batch_size=4, sampler=val_sampler)

print("train indices used:", sorted(train_indices))
print("val indices used:", sorted(val_indices))
print("\ntotal samples seen across train_loader:", sum(len(b[0]) for b in train_loader))
print("total samples seen across val_loader:", sum(len(b[0]) for b in val_loader))


train indices used: [1, 2, 4, 5, 6, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
val indices used: [0, 3, 7, 8]

total samples seen across train_loader: 16
total samples seen across val_loader: 4


### 7. WeightedRandomSampler — oversampling rare classes

**What this cell does (plain language):** we build a deliberately imbalanced
dataset (mostly class 0, few class 1), then use `WeightedRandomSampler` to
make the DataLoader sample the rare class more often, and count how often
each class appears in a full pass to prove it's working.


In [7]:
from torch.utils.data import WeightedRandomSampler
from collections import Counter

# imbalanced: 90 samples of class 0, 10 samples of class 1
labels = [0] * 90 + [1] * 10
imbalanced_dataset = NumberDataset(list(range(100)), labels)

class_counts = Counter(labels)
print("class counts in the raw dataset:", dict(class_counts))

# weight each SAMPLE inversely proportional to its class frequency
sample_weights = [1.0 / class_counts[label] for label in labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=100, replacement=True)

weighted_loader = DataLoader(imbalanced_dataset, batch_size=10, sampler=sampler)
seen_labels = []
for _, batch_y in weighted_loader:
    seen_labels.extend(batch_y.tolist())

print("class counts after weighted sampling (should be much closer to 50/50):", Counter(seen_labels))


class counts in the raw dataset: {0: 90, 1: 10}
class counts after weighted sampling (should be much closer to 50/50): Counter({1: 57, 0: 43})


### 9. num_workers — parallel data loading

**What this cell does (plain language):** we time loading the same dataset
with `num_workers=0` (single process) vs a higher value, on a dataset with
an artificial delay per sample, to demonstrate why `num_workers` matters for
real (slow-to-load) data like images read from disk.


In [8]:
import time

class SlowDataset(Dataset):
    """Simulates slow disk/decode work per sample with a tiny sleep."""
    def __init__(self, n):
        self.n = n
    def __len__(self):
        return self.n
    def __getitem__(self, idx):
        time.sleep(0.01)  # pretend this is image decoding / disk I/O
        return torch.tensor([idx], dtype=torch.float32)

slow_dataset = SlowDataset(100)

start = time.time()
loader_0_workers = DataLoader(slow_dataset, batch_size=10, num_workers=0)
for _ in loader_0_workers:
    pass
print(f"num_workers=0 took: {time.time() - start:.2f}s")

# NOTE: num_workers > 0 spins up subprocesses, which has its own overhead -
# for small/fast datasets like this toy example it can even be SLOWER due to
# process startup cost. The benefit shows up with genuinely slow per-sample
# loading (large images, decoding, augmentation) combined with many samples.
# We keep this example at num_workers=0 for reliability in this notebook
# environment; try num_workers=2 or 4 locally on a real, larger dataset.


num_workers=0 took: 1.02s


### 10. Applying transforms inside __getitem__

**What this cell does (plain language):** the standard pattern for data
augmentation — do the transform work lazily, per sample, inside
`__getitem__`, rather than transforming the whole dataset upfront. This
matters because with `shuffle=True` and random augmentations, every epoch
sees a *freshly randomized* version of each sample.


In [ ]:
class AugmentedDataset(Dataset):
    def __init__(self, data):
        self.data = data  # plain list of numbers, standing in for "raw images"

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = torch.tensor([self.data[idx]], dtype=torch.float32)
        # a toy "augmentation": add small random noise, freshly each call
        noise = torch.randn_like(x) * 0.1
        return x + noise

aug_dataset = AugmentedDataset([10.0, 20.0, 30.0])
print("same index (0), fetched twice - values differ due to fresh random noise each time:")
print("call 1:", aug_dataset[0])
print("call 2:", aug_dataset[0])

# This is why augmentation belongs in __getitem__, not precomputed once in
# __init__ - precomputing would "freeze" one random version instead of
# giving a new one every epoch, which is far less effective for training.


same index (0), fetched twice - values differ due to fresh random noise each time:
call 1: tensor([10.0469])
call 2: tensor([9.9894])


## Common bugs

- **`RuntimeError: stack expects each tensor to be equal size` in the default collate**
  Your samples have variable shapes/lengths (e.g. different-length
  sentences) and you didn't provide a custom `collate_fn`. Fix: write a
  `collate_fn` that pads (or otherwise combines) variable-length samples
  before stacking, as shown in section 4.

- **Data isn't actually shuffled every epoch**
  Created the `DataLoader` with `shuffle=False` for training by mistake, or
  reused a fixed `sampler` object incorrectly. Fix: `shuffle=True` for
  training, `False` for validation/test; if using a custom sampler, make sure
  it reshuffles internally (like `RandomSampler` does) rather than fixing order.

- **Training loop hangs or is extremely slow with num_workers > 0**
  Common on certain OS/environment combinations (notably some Windows/Jupyter
  setups) due to how worker processes are spawned. Fix: try `num_workers=0`
  first to confirm the pipeline itself works, then increase gradually; on
  problematic setups, guard your script's entry point with
  `if __name__ == "__main__":` as multiprocessing requires.

- **`sampler` and `shuffle=True` used together**
  `ValueError: sampler option is mutually exclusive with shuffle`. Fix: use
  one or the other — pass a `sampler=` object for custom sampling logic, or
  `shuffle=True/False` for the simple built-in case, never both.

- **Last batch has a different (smaller) size and breaks the training step**
  Some code assumes every batch has exactly `batch_size` samples (e.g. a
  fixed-size buffer, or certain distributed/BatchNorm configurations). Fix:
  set `drop_last=True` if a smaller final batch would break your training step.

- **Augmentation applied once instead of freshly every epoch**
  Precomputing augmented versions of the data in `__init__` "freezes" one
  random variant instead of generating a new one each time a sample is
  fetched. Fix: do randomized transforms inside `__getitem__`, not `__init__`.

- **Forgot to move batches to the correct device in the training loop**
  `DataLoader` output stays wherever the underlying tensors were created
  (typically CPU) — it does NOT automatically move data to GPU. Fix:
  `inputs, labels = inputs.to(device), labels.to(device)` inside the loop,
  every batch.

- **Using an `IterableDataset` but expecting `len()` / shuffling to work**
  `IterableDataset` has no built-in concept of length or random access —
  `shuffle=True` has no effect and `len(loader)` may error or mislead. Fix:
  implement your own buffering/shuffling logic inside `__iter__` if you need
  approximate shuffling for a streaming dataset.


## Exercises

Attempt each one in the empty cell right after the question. My full solution is in the cell after that - don't peek until you've tried.

**Exercise 1 - Build a custom Dataset.**
Write a `SquareDataset` that, given a list of numbers, returns
`(number, number**2)` pairs as `(x, y)` tensors from `__getitem__`. Verify
with `len()` and by indexing a couple of items.

In [ ]:
# Your attempt for Exercise 1 here\n

**Solution 1**

In [11]:
# Solution 1
import torch
from torch.utils.data import Dataset

class SquareDataset(Dataset):
    def __init__(self, numbers):
        self.numbers = numbers
    def __len__(self):
        return len(self.numbers)
    def __getitem__(self, idx):
        n = self.numbers[idx]
        return torch.tensor([n], dtype=torch.float32), torch.tensor([n**2], dtype=torch.float32)

ds = SquareDataset([1, 2, 3, 4])
print("len:", len(ds))
print("ds[0]:", ds[0])
print("ds[3]:", ds[3])


len: 4
ds[0]: (tensor([1.]), tensor([1.]))
ds[3]: (tensor([4.]), tensor([16.]))


**Exercise 2 - Batch size math.**
Given a dataset of 37 samples and `batch_size=8`, how many batches will a
`DataLoader` produce with `drop_last=False`? With `drop_last=True`? What is
the size of the last batch in each case? Verify with code.

In [12]:
# Your attempt for Exercise 2 here\n

**Solution 2**

In [13]:
# Solution 2
# 37 samples, batch_size=8 -> 37/8 = 4 full batches (32 samples) + 1 leftover batch of 5
# drop_last=False -> 5 batches total, last one has 5 samples
# drop_last=True  -> 4 batches total, last one still has 8 (the partial one is dropped)

import torch
from torch.utils.data import Dataset, DataLoader

class DummyDataset(Dataset):
    def __init__(self, n): self.n = n
    def __len__(self): return self.n
    def __getitem__(self, idx): return torch.tensor([idx])

ds = DummyDataset(37)
loader_keep = DataLoader(ds, batch_size=8, drop_last=False)
loader_drop = DataLoader(ds, batch_size=8, drop_last=True)

print("drop_last=False -> num batches:", len(loader_keep), "| last batch size:", len(list(loader_keep)[-1]))
print("drop_last=True  -> num batches:", len(loader_drop), "| last batch size:", len(list(loader_drop)[-1]))


drop_last=False -> num batches: 5 | last batch size: 5
drop_last=True  -> num batches: 4 | last batch size: 8


**Exercise 3 - Write a collate_fn for images + variable-length captions.**
Assume each sample is `(image_tensor, caption_list_of_token_ids)` where every
image is the same fixed shape `(3, 32, 32)` but captions vary in length.
Write a `collate_fn` that stacks the images normally but pads the captions.

In [ ]:
# Your attempt for Exercise 3 here\n

**Solution 3**

In [15]:
# Solution 3
import torch

def image_caption_collate(batch):
    # batch: list of (image_tensor, caption_list) tuples
    images = torch.stack([item[0] for item in batch])  # all same shape, stacks fine

    captions = [torch.tensor(item[1], dtype=torch.long) for item in batch]
    lengths = [len(c) for c in captions]
    max_len = max(lengths)
    padded_captions = torch.zeros(len(batch), max_len, dtype=torch.long)
    for i, cap in enumerate(captions):
        padded_captions[i, :len(cap)] = cap

    return images, padded_captions, torch.tensor(lengths)

# quick test
fake_batch = [
    (torch.randn(3, 32, 32), [1, 2, 3]),
    (torch.randn(3, 32, 32), [4, 5]),
]
images, captions, lengths = image_caption_collate(fake_batch)
print("images shape:", images.shape)
print("captions:\n", captions)
print("lengths:", lengths)


images shape: torch.Size([2, 3, 32, 32])
captions:
 tensor([[1, 2, 3],
        [4, 5, 0]])
lengths: tensor([3, 2])


**Exercise 4 - Reproduce the "mutually exclusive" error.**
Write code that passes both `shuffle=True` and a `sampler=` to `DataLoader`
at the same time, catch the resulting error, and print its message.

In [ ]:
# Your attempt for Exercise 4 here\n

**Solution 4**

In [17]:
# Solution 4
import torch
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler

class DummyDataset(Dataset):
    def __init__(self, n): self.n = n
    def __len__(self): return self.n
    def __getitem__(self, idx): return torch.tensor([idx])

ds = DummyDataset(10)
sampler = SubsetRandomSampler(list(range(5)))

try:
    loader = DataLoader(ds, batch_size=2, shuffle=True, sampler=sampler)
except ValueError as e:
    print("Error:", e)


Error: sampler option is mutually exclusive with shuffle


**Exercise 5 - Train/val split by percentage.**
Write a reusable function `train_val_split(dataset, val_fraction=0.2, seed=0)`
that returns `(train_loader, val_loader)` using `SubsetRandomSampler`, given
any map-style `Dataset` and a `batch_size`.

In [ ]:
# Your attempt for Exercise 5 here\n

**Solution 5**

In [19]:
# Solution 5
import random
from torch.utils.data import DataLoader, SubsetRandomSampler

def train_val_split(dataset, batch_size, val_fraction=0.2, seed=0):
    indices = list(range(len(dataset)))
    random.Random(seed).shuffle(indices)
    split = int((1 - val_fraction) * len(indices))
    train_idx, val_idx = indices[:split], indices[split:]

    train_loader = DataLoader(dataset, batch_size=batch_size, sampler=SubsetRandomSampler(train_idx))
    val_loader = DataLoader(dataset, batch_size=batch_size, sampler=SubsetRandomSampler(val_idx))
    return train_loader, val_loader

ds = DummyDataset(50)
train_loader, val_loader = train_val_split(ds, batch_size=4, val_fraction=0.2, seed=0)
print("train samples:", sum(len(b) for b in train_loader))
print("val samples:", sum(len(b) for b in val_loader))


train samples: 40
val samples: 10


**Exercise 6 - Oversampling by hand.**
Given `labels = [0]*80 + [1]*15 + [2]*5` (3 imbalanced classes), compute the
per-sample weights needed for a `WeightedRandomSampler` so all 3 classes are
sampled roughly equally, and verify with a `Counter` after sampling.

In [ ]:
# Your attempt for Exercise 6 here\n

**Solution 6**

In [21]:
# Solution 6
from collections import Counter
from torch.utils.data import WeightedRandomSampler, DataLoader

labels = [0]*80 + [1]*15 + [2]*5
class_counts = Counter(labels)
print("original counts:", dict(class_counts))

weights = [1.0 / class_counts[label] for label in labels]
sampler = WeightedRandomSampler(weights, num_samples=len(labels), replacement=True)

ds = DummyDataset(len(labels))
loader = DataLoader(ds, batch_size=10, sampler=sampler)

seen = []
for batch in loader:
    for idx in batch.squeeze(-1).tolist():
        seen.append(labels[int(idx)])
print("counts after weighted sampling (should be closer to equal):", Counter(seen))


original counts: {0: 80, 1: 15, 2: 5}
counts after weighted sampling (should be closer to equal): Counter({2: 39, 0: 34, 1: 27})


**Exercise 7 - Spot the augmentation bug.**
This dataset precomputes an "augmented" version once and reuses it forever
instead of re-randomizing every access. Rewrite it correctly:
```python
class BuggyAugmented(Dataset):
    def __init__(self, data):
        self.data = [x + torch.randn(1).item() * 0.1 for x in data]  # bug is here
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return torch.tensor([self.data[idx]])
```

In [ ]:
# Your attempt for Exercise 7 here\n

**Solution 7**

In [23]:
# Solution 7
import torch
from torch.utils.data import Dataset

class FixedAugmented(Dataset):
    def __init__(self, data):
        self.data = data  # store the ORIGINAL data, don't precompute noise

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # apply randomness HERE, fresh every call
        noise = torch.randn(1).item() * 0.1
        return torch.tensor([self.data[idx] + noise])

ds = FixedAugmented([1.0, 2.0, 3.0])
print("call 1:", ds[0])
print("call 2:", ds[0], " <- different now, correctly re-randomized each access")


call 1: tensor([1.0338])
call 2: tensor([1.1016])  <- different now, correctly re-randomized each access


**Exercise 8 - drop_last with a fixed-size assumption.**
Write a tiny training loop over a dataset of 22 samples with `batch_size=5`
that would break if it assumed every batch had exactly 5 samples (e.g. by
reshaping to a fixed shape). Show the fix using `drop_last=True`.

In [ ]:
# Your attempt for Exercise 8 here\n

**Solution 8**

In [25]:
# Solution 8
import torch
from torch.utils.data import Dataset, DataLoader

class DummyDataset(Dataset):
    def __init__(self, n): self.n = n
    def __len__(self): return self.n
    def __getitem__(self, idx): return torch.tensor([float(idx)])

ds = DummyDataset(22)

# WITHOUT drop_last: last batch has only 2 samples, breaks a fixed reshape
loader_buggy = DataLoader(ds, batch_size=5, drop_last=False)
try:
    for batch in loader_buggy:
        reshaped = batch.view(5, 1)  # assumes exactly 5 samples - fails on the last batch
except RuntimeError as e:
    print("Broke as expected on the partial batch:", str(e)[:100])

# FIX: drop_last=True guarantees every batch has exactly batch_size samples
loader_fixed = DataLoader(ds, batch_size=5, drop_last=True)
for batch in loader_fixed:
    reshaped = batch.view(5, 1)  # always safe now
print("Fixed version completed without error. Total batches:", len(loader_fixed))


Broke as expected on the partial batch: shape '[5, 1]' is invalid for input of size 2
Fixed version completed without error. Total batches: 4


**Exercise 9 - Moving batches to device.**
Write the standard training-loop snippet that correctly moves both the
inputs and labels from a `DataLoader` batch onto a `device` variable, for a
loader that yields `(x, y)` tuples.

In [ ]:
# Your attempt for Exercise 9 here\n

**Solution 9**

In [27]:
# Solution 9
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# standard training-loop snippet
# for x, y in data_loader:
#     x, y = x.to(device), y.to(device)
#     ...forward/backward pass...

# demonstrated concretely:
x = torch.randn(4, 10)
y = torch.randint(0, 2, (4,))
x, y = x.to(device), y.to(device)
print("x device:", x.device, "| y device:", y.device)


x device: cuda:0 | y device: cuda:0


**Exercise 10 - IterableDataset basics.**
Write a minimal `IterableDataset` subclass that yields squares of numbers
from 0 up to some `n`, using `__iter__`, and show it working inside a
`DataLoader` with `batch_size=4`.

In [ ]:
# Your attempt for Exercise 10 here\n

**Solution 10**

In [ ]:
# Solution 10
import torch
from torch.utils.data import IterableDataset, DataLoader

class SquareStream(IterableDataset):
    def __init__(self, n):
        self.n = n

    def __iter__(self):
        for i in range(self.n):
            yield torch.tensor([i ** 2], dtype=torch.float32)

stream_ds = SquareStream(10)
# note: shuffle= is not supported/meaningful for IterableDataset
loader = DataLoader(stream_ds, batch_size=4)
for batch in loader:
    print(batch.squeeze().tolist())


[0.0, 1.0, 4.0, 9.0]
[16.0, 25.0, 36.0, 49.0]
[64.0, 81.0]
